# Campaign-23 test-fragment inference viewer

Loads the selected Campaign 23 architecture and inference geometry, then renders each test fragment and the w055 holdout.

Each full-size figure contains:

| raw inference | inference | fiber composite | fiber composite with scale bar |

- the checkpoint is loaded strictly after removing training-only heads
- inference uses one plain pass through the shared bounded-memory row reader
- inference and composites are restricted to the tile-aligned nonzero data bounds
- the visualizer module is reloaded so stale notebook kernels cannot retain an older implementation
- context, depth, multitile, and model settings come from the saved run config
- composites are cached separately because they are expensive to generate
- set `DRY_RUN = True` in cell 2 to inspect composites without loading the model

In [ ]:
# campaign-23 literal-surface slice-8 run
import json
import os
from pathlib import Path

DRY_RUN = False
EXP_NAME = os.getenv("VESUVIUS_EXP_NAME", "23_baseline")
RUN_ROOT = Path(os.getenv("VESUVIUS_RUN_ROOT", "/vesuvius/runs_archs23"))
RUN_ID = os.getenv("VESUVIUS_RUN_ID", "")
if RUN_ID:
    RUN_DIR = RUN_ROOT / RUN_ID
else:
    candidates = sorted(RUN_ROOT.glob(f"{EXP_NAME}_*/config.json"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(f"no completed run config matching {EXP_NAME!r} under {RUN_ROOT}")
    RUN_DIR = candidates[-1].parent
    RUN_ID = RUN_DIR.name
RUN_CONFIG_PATH = RUN_DIR / "config.json"
MODEL_PATH = Path(os.getenv(
    "VESUVIUS_MODEL_PATH",
    "/vesuvius/models/archs23/baseline_best_character.pth",
))
if not MODEL_PATH.exists():
    MODEL_PATH = Path("/vesuvius/models/archs23/baseline.pth")

with open(RUN_CONFIG_PATH, "r", encoding="utf-8") as f:
    RUN_CONFIG = json.load(f)
_RUN_DATA = RUN_CONFIG["data"]
_RUN_MODEL = RUN_CONFIG["model"]
ARCH = _RUN_MODEL["arch"]
TILE_SIZE = int(_RUN_DATA["tile_size"])
CONTEXT_SIZE = int(_RUN_DATA["context_size"])
CONTEXT_DOWNSAMPLE = int(_RUN_DATA["context_downsample"])
DEPTH = int(_RUN_DATA["depth"])
D_START = int(_RUN_DATA["d_start"])
D_END = int(_RUN_DATA["d_end"])
TTA = False
TTA_MODE = _RUN_DATA["tta_mode"]
INFER_BS = int(_RUN_DATA["eval_infer_bs"])
EVAL_PREFETCH = int(_RUN_DATA["eval_prefetch"])
EVAL_CHUNK_GB = float(_RUN_DATA["eval_chunk_gb"])
PRELOAD_RAM = True
TORCH_COMPILE = False
COMPOSITE_METHOD = "maxproj"
COMPOSITE_D0 = 10
COMPOSITE_D1 = 18
COMPOSITE_DISPLAY = "raw"
COMPOSITE_CLAHE = False
GENERATE_COMPOSITES = False
VOXEL_UM = 9.362
DISPLAY_SCALE = 0.25
MASK_CROP_MARGIN = 32
PAD_PX = 24
OUTPUT_DIR = "output"
COMPOSITE_CACHE = "output/composite_cache"
FRAGMENTS = {
    "auto_grown_20260814140748": 20260814140748,
    "auto_grown_20260717193517": 20260717193517,
    "auto_grown_20260720090842": 20260720090842,
    "auto_grown_20250703034159": 20250703034159,
    "auto_grown_20260723112922652_merged": 20260723112922,
}
if DEPTH != 8 or not _RUN_MODEL.get("surface_teacher_input") or not _RUN_DATA.get("surface_relative_depth_window"):
    raise RuntimeError("selected run is not literal-surface true slice-8")
print(RUN_ID, MODEL_PATH, f"depth={DEPTH}", "literal surface", f"ctx={CONTEXT_SIZE}/ds{CONTEXT_DOWNSAMPLE}", f"test_scrolls={len(FRAGMENTS)}")

20_bce_soft_noweight_5090_09_20-59-17 | nnunet3d_lcndz | tile=16 | ctx=192/ds2 | center=64


In [7]:
import gc
import importlib
import os
import sys
import types

REPO = os.getenv("VESUVIUS_REPO", "/vesuvius" if os.name == "posix" else r"C:\Users\ChenJeff\Documents\vesuvius")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

from utils.config import Config
from utils.dataloader import _load_unified_cache
from utils.model import create_model
from utils import visualizer as visualizer_module

# reload the bounded-memory row reader in existing kernels
visualizer_module = importlib.reload(visualizer_module)
TBV = visualizer_module.TensorboardVisualizer
group_by_depth = visualizer_module.group_by_depth
predict_tiles = visualizer_module.predict_tiles
load_or_create_midslice_mask = visualizer_module.load_or_create_midslice_mask

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(COMPOSITE_CACHE, exist_ok=True)
print(
    f"repo={REPO}",
    f"| torch={torch.__version__}",
    f"| cuda={torch.cuda.is_available()}",
    f"| dry_run={DRY_RUN}",
    "| bounded-memory visualizer reloaded",
)

repo=/vesuvius | torch=2.11.0+cu128 | cuda=True | dry_run=False | bounded-memory visualizer reloaded


In [ ]:
# composite helper and projection settings
def _project_depth(vol, d0, d1, method, bounds=None):
    """project only the requested spatial bounds over slices [d0, d1)"""
    _, height, width = map(int, vol.shape)
    y0, y1, x0, x1 = bounds or (0, height, 0, width)
    d0, d1 = max(0, d0), min(d1, int(vol.shape[0]))
    shape = (y1 - y0, x1 - x0)
    if method == "maxproj":
        acc = np.zeros(shape, np.float32)
        for depth in range(d0, d1):
            acc = np.maximum(acc, np.asarray(vol[depth, y0:y1, x0:x1]).astype(np.float32))
        return acc
    if method == "meanproj":
        acc = np.zeros(shape, np.float64)
        for depth in range(d0, d1):
            acc += np.asarray(vol[depth, y0:y1, x0:x1])
        return (acc / max(d1 - d0, 1)).astype(np.float32)
    if method == "minproj":
        acc = np.full(shape, np.inf, np.float32)
        for depth in range(d0, d1):
            acc = np.minimum(acc, np.asarray(vol[depth, y0:y1, x0:x1]).astype(np.float32))
        return np.where(np.isfinite(acc), acc, 0).astype(np.float32)
    if method == "stdproj":
        total = np.zeros(shape, np.float64)
        total_squared = np.zeros(shape, np.float64)
        for depth in range(d0, d1):
            layer = np.asarray(vol[depth, y0:y1, x0:x1]).astype(np.float64)
            total += layer
            total_squared += layer * layer
        count = max(d1 - d0, 1)
        mean = total / count
        return np.sqrt(np.clip(total_squared / count - mean * mean, 0, None)).astype(np.float32)
    if method == "midslice":
        return np.asarray(vol[(d0 + d1) // 2, y0:y1, x0:x1]).astype(np.float32)
    raise ValueError(f"unknown composite method: {method}")


def _display_map(proj, mask, mode, use_clahe):
    """map a float projection to uint8 for display"""
    if mode == "raw":
        image = np.clip(proj, 0, 255).astype(np.uint8)
    else:
        lo, hi = np.percentile(proj[mask], [1, 99]) if mask.any() else (float(proj.min()), float(proj.max()))
        display = np.clip((proj - lo) / max(hi - lo, 1e-6), 0, 1)
        image = (display * 255).astype(np.uint8)
    if use_clahe:
        image = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image)
    return image * mask.astype(np.uint8)


def make_composite(scroll_id, bounds, method=None, d0=None, d1=None, use_cache=True, display=None, clahe=None):
    """return a uint8 fiber composite restricted to the data bbox"""
    method = method or COMPOSITE_METHOD
    display = display or COMPOSITE_DISPLAY
    clahe = COMPOSITE_CLAHE if clahe is None else clahe
    vol = zarr.open(os.path.join(Config().data.zarr_path, f"{scroll_id}.zarr"), mode="r")
    depth_count = int(vol.shape[0])
    d0 = COMPOSITE_D0 if d0 is None else d0
    d1 = (COMPOSITE_D1 if COMPOSITE_D1 is not None else depth_count) if d1 is None else d1
    d0, d1 = max(0, d0), min(d1, depth_count)
    y0, y1, x0, x1 = bounds
    bounds_tag = f"y{y0}-{y1}_x{x0}-{x1}"
    display_tag = f"_{display}" + ("_clahe" if clahe else "")
    cache_path = os.path.join(
        COMPOSITE_CACHE,
        f"{scroll_id}_{method}_{d0}-{d1}_{bounds_tag}{display_tag}.png",
    )
    if use_cache and not GENERATE_COMPOSITES and os.path.exists(cache_path):
        return np.array(Image.open(cache_path).convert("L"))
    mask_path = f"masks/{scroll_id}.png"
    full_mask = (np.array(Image.open(mask_path).convert("L")) > 0) if os.path.exists(mask_path) else None
    proj = _project_depth(vol, d0, d1, method, bounds=bounds)
    mask = full_mask[y0:y1, x0:x1] if full_mask is not None else (proj > 0)
    image = _display_map(proj, mask, display, clahe)
    Image.fromarray(image).save(cache_path)
    print(f"[composite] {scroll_id} {method} d{d0}-{d1} {bounds_tag}{display_tag} -> {cache_path} {image.shape}")
    return image

In [ ]:
# inference and reusable figure builder
_AUXILIARY_PREFIXES = ("supcon_head.", "domain_head.")


def build_config():
    config = Config()
    for name in (
        "tile_size", "depth", "context_size", "context_downsample", "d_start", "d_end",
        "train_d_start", "train_d_end", "surface_label_dir", "surface_relative_depth_window",
        "eval_infer_bs", "eval_prefetch", "eval_chunk_gb", "tta_mode",
    ):
        if name in _RUN_DATA and hasattr(config.data, name):
            setattr(config.data, name, _RUN_DATA[name])
    for name, value in _RUN_MODEL.items():
        if hasattr(config.model, name):
            setattr(config.model, name, value)
    config.model.compile_model = False
    config.tra.supcon = False
    config.tra.dann = False
    config.device = "cuda" if torch.cuda.is_available() else "cpu"
    return config


def _checkpoint_state(path):
    state = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    cleaned = {}
    for key, value in state.items():
        key = key.removeprefix("module.").removeprefix("_orig_mod.")
        if not key.startswith(_AUXILIARY_PREFIXES):
            cleaned[key] = value
    return cleaned, len(state) - len(cleaned)


def load_model(config):
    model, parameter_count = create_model(config)
    state, ignored = _checkpoint_state(MODEL_PATH)
    model.load_state_dict(state, strict=True)
    model.eval()
    if TORCH_COMPILE:
        model = torch.compile(model, mode="reduce-overhead")
    print(f"loaded {MODEL_PATH} params={parameter_count:,} ignored_training_only={ignored}")
    return model


def get_norm(scroll_id, vol, mask):
    stats = _load_unified_cache().get(str(scroll_id))
    if stats and all(key in stats for key in ("mean", "std", "min", "max")):
        return stats["mean"], stats["std"], stats["min"], stats["max"]
    raise RuntimeError(f"normalization cache missing for {scroll_id}")


def _mask_bbox(mask_bool, margin, align):
    """return an outward tile-aligned bbox around nonzero data"""
    ys, xs = np.where(mask_bool)
    if ys.size == 0:
        return 0, mask_bool.shape[0], 0, mask_bool.shape[1]
    align = max(1, int(align))
    y0 = max(0, ((int(ys.min()) - margin) // align) * align)
    x0 = max(0, ((int(xs.min()) - margin) // align) * align)
    y1 = min(mask_bool.shape[0], ((int(ys.max()) + 1 + margin + align - 1) // align) * align)
    x1 = min(mask_bool.shape[1], ((int(xs.max()) + 1 + margin + align - 1) // align) * align)
    return y0, y1, x0, x1


def predict_map(model, config, scroll_id, bounds):
    """predict one plain map inside the requested data bounds"""
    import time as _time

    source = zarr.open(os.path.join(config.data.zarr_path, f"{scroll_id}.zarr"), mode="r")
    mask_full = np.array(Image.open(f"masks/{scroll_id}.png").convert("L")) > 0
    y0, y1, x0, x1 = bounds
    if PRELOAD_RAM:
        t0 = _time.time()
        print(f"[preload] {scroll_id} bbox={bounds} -> RAM ... ", end="", flush=True)
        vol = np.asarray(source[:, y0:y1, x0:x1])
        mask = mask_full[y0:y1, x0:x1]
        y_range, x_range = (0, y1 - y0), (0, x1 - x0)
        print(f"done in {_time.time() - t0:.1f}s", flush=True)
    else:
        vol = source
        mask = mask_full
        y_range, x_range = (y0, y1), (x0, x1)
    mean, std, global_min, global_max = get_norm(scroll_id, vol, mask)
    surface_dir = os.path.join(config.data.surface_label_dir, str(scroll_id))
    surface_depth = np.load(os.path.join(surface_dir, "depth.npy"), mmap_mode="r")
    surface_confidence = np.load(os.path.join(surface_dir, "confidence.npy"), mmap_mode="r")
    if PRELOAD_RAM:
        surface_depth = surface_depth[y0:y1, x0:x1]
        surface_confidence = surface_confidence[y0:y1, x0:x1]
    fake = types.SimpleNamespace(c=config)
    coords = TBV._gen_tile_coords(
        fake,
        (config.data.d_start, config.data.d_end),
        y_range,
        x_range,
        mask,
        z_step=config.data.depth,
    )
    grouped = group_by_depth(coords)
    if len(grouped) != 1:
        raise RuntimeError(f"surface-relative inference expected one depth pass, got {len(grouped)}")
    depth_offset = next(iter(grouped))
    return predict_tiles(
        config,
        model,
        vol,
        mask,
        grouped[depth_offset],
        y_range,
        x_range,
        config.data.d_start + depth_offset,
        f"test_{scroll_id}",
        mean,
        std,
        global_min,
        global_max,
        surface_depth_map=surface_depth,
        surface_confidence_map=surface_confidence,
    )


def _colorize(pmap, out_hw):
    valid = np.isfinite(pmap)
    p8 = (np.clip(np.nan_to_num(pmap, nan=0.0), 0, 1) * 255).astype(np.uint8)
    bgr = cv2.applyColorMap(p8, cv2.COLORMAP_INFERNO)
    bgr[~valid] = (115, 115, 115)
    return cv2.resize(bgr, (out_hw[1], out_hw[0]), interpolation=cv2.INTER_NEAREST)


def _red_scale_bar_1cm(bgr):
    height, width = bgr.shape[:2]
    length = int(round(10000.0 / VOXEL_UM))
    red = (0, 0, 255)
    thickness = max(4, min(height, width) // 250)
    pad = int(0.05 * min(height, width)) + thickness
    if width >= height:
        x1, y = width - pad, height - pad
        x0 = max(0, x1 - length)
        cv2.rectangle(bgr, (x0, y - thickness), (x1, y), red, -1)
        cv2.putText(bgr, "1 cm", (x0, y - thickness - 12), cv2.FONT_HERSHEY_SIMPLEX, 1.8, red, 3, cv2.LINE_AA)
    else:
        y1, x = height - pad, width - pad
        y0 = max(0, y1 - length)
        cv2.rectangle(bgr, (x - thickness, y0), (x, y1), red, -1)
        cv2.putText(bgr, "1 cm", (max(0, x - 160), max(20, y0 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 1.8, red, 3, cv2.LINE_AA)
    return bgr


def _pad(panel):
    return cv2.copyMakeBorder(panel, PAD_PX, PAD_PX, PAD_PX, PAD_PX, cv2.BORDER_CONSTANT, value=(255, 255, 255))


def _run_name():
    return RUN_ID


def render_fragment(name, scroll_id, model=None, config=None):
    mask_full = np.array(Image.open(f"masks/{scroll_id}.png").convert("L"))
    bounds = _mask_bbox(mask_full > 0, MASK_CROP_MARGIN, TILE_SIZE)
    y0, y1, x0, x1 = bounds
    crop_height, crop_width = y1 - y0, x1 - x0
    composite = make_composite(scroll_id, bounds=bounds)
    if composite.shape != (crop_height, crop_width):
        composite = cv2.resize(composite, (crop_width, crop_height), interpolation=cv2.INTER_AREA)
    composite_panel = cv2.cvtColor(composite, cv2.COLOR_GRAY2BGR)
    if DRY_RUN or model is None:
        panels = [composite_panel.copy() for _ in range(4)]
        kind = "composite dry run"
    else:
        prediction = predict_map(model, config, scroll_id, bounds)
        prediction_panel = _colorize(prediction, (crop_height, crop_width))
        panels = [
            prediction_panel.copy(),
            prediction_panel.copy(),
            composite_panel.copy(),
            composite_panel.copy(),
        ]
        kind = "plain inference | inference | composite | scale"
    panels[3] = _red_scale_bar_1cm(panels[3])
    panels = [_pad(panel) for panel in panels]
    big = np.hstack(panels) if crop_height >= crop_width else np.vstack(panels)
    out_dir = os.path.join(OUTPUT_DIR, "test_visualizations", _run_name())
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{name}.jpg")
    if not cv2.imwrite(out_path, big, [cv2.IMWRITE_JPEG_QUALITY, 92]):
        raise RuntimeError(f"failed to save {out_path}")
    print(f"[saved] {out_path} ({kind}) bbox={bounds}")
    del big, panels
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [8]:
# build the exact run architecture and load once
from utils.platform import get_zarr_dir

C = build_config()
C.data.zarr_path = get_zarr_dir()
MODEL = None if DRY_RUN else load_model(C)


def ensure_fragment_mask(scroll_id):
    """refresh the fragment mask from the center layer of its active zarr"""
    zarr_path = os.path.join(C.data.zarr_path, f"{scroll_id}.zarr")
    if not os.path.isdir(zarr_path):
        raise FileNotFoundError(
            f"test zarr is missing from the active zarr directory: {zarr_path}"
        )
    volume = zarr.open(zarr_path, mode="r")
    return load_or_create_midslice_mask(
        volume, os.path.join(REPO, "masks", f"{scroll_id}.png"), refresh=True
    )


print(
    "model:", "skipped" if MODEL is None else C.model.arch,
    "| zarr:", C.data.zarr_path,
    "| context:", C.data.context_size,
    "ds", C.data.context_downsample,
    "| feature_attn_mil:", C.model.feature_attn_mil,
    "| learned_surface:", C.model.learned_surface,
    "| multitile:", f"{C.model.multitile_subtile}x{C.model.multitile_grid}",
)

Model parameters (nnunet3d_lcndz): 5,609,085
loaded /vesuvius/models/archs20/bce_soft_noweight_5090_best_character.pth  params=5,609,085  ignored_training_only=4
model: nnunet3d_lcndz | zarr: /vesuvius/ves_zarrs2 | context: 192 ds 2 | feature_attn_mil: True | learned_surface: True | multitile: 16x4


In [ ]:
# --- test fragment 1: PHerc0813 (updated patch, 33.31cm², 5081×5701) ---
NAME = "auto_grown_20260814140748"
ensure_fragment_mask(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)

FileNotFoundError: [Errno 2] No such file or directory: 'masks/20260814140748.png'

In [ ]:
# --- test fragment 2: PHerc0211 merged (5 patches, 7181×6501) ---
NAME = "auto_grown_20260717193517"
ensure_fragment_mask(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)

In [ ]:
# --- test fragment 4: PHerc1447 ---
# 51.27cm^2, 6264×8318 (8.64µm src, upsampled to 9.4µm)
NAME = "auto_grown_20250703034159"
ensure_fragment_mask(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)

In [ ]:
# --- test fragment 3: PHerc1203 ---
# 7.9cm^2, 4035×4455
NAME = "auto_grown_20260720090842"
ensure_fragment_mask(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)

In [ ]:
# test fragment 5: PHerc0826 merged, cropped to rendered data bounds
NAME = "auto_grown_20260723112922652_merged"
ensure_fragment_mask(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)

In [ ]:
# w055 remains a separate holdout and is not part of this test pass
print("[skip] w055 holdout")